In [2]:
!pip install ultralytics
import ultralytics
ultralytics.checks()

Ultralytics 8.3.124 🚀 Python-3.11.12 torch-2.6.0+cu124 CPU (Intel Xeon 2.20GHz)
Setup complete ✅ (2 CPUs, 12.7 GB RAM, 41.1/107.7 GB disk)


In [3]:
from ultralytics import YOLO
import cv2
import os
from google.colab import files
from IPython.display import Image, display

# Load the pre-trained YOLOv8 model
model = YOLO("yolov8n.pt")  # Nano model for faster inference; use yolov8s.pt, yolov8m.p

100%|██████████| 6.25M/6.25M [00:00<00:00, 67.2MB/s]


In [4]:
uploaded = files.upload()

Saving cars-moving.mp4 to cars-moving.mp4


In [5]:
video_path = list(uploaded.keys())[0]

In [6]:
# Define output video path
output_path = 'output_video.mp4'

# Run object detection on the video
results = model.predict(source=video_path, save=True, project="runs/detect", name="exp", exist_ok=True)

# The output video is saved in the runs/detect/exp folder
# Find the output video path (YOLOv8 appends a suffix to the saved file)
output_video_path = f"runs/detect/exp/{video_path.split('/')[-1]}"


WARNING ⚠️ 
inference results will accumulate in RAM unless `stream=True` is passed, causing potential out-of-memory
errors for large sources or long-running streams and videos. See https://docs.ultralytics.com/modes/predict/ for help.

Example:
    results = model(source=..., stream=True)  # generator of Results objects
    for r in results:
        boxes = r.boxes  # Boxes object for bbox outputs
        masks = r.masks  # Masks object for segment masks outputs
        probs = r.probs  # Class probabilities for classification outputs

video 1/1 (frame 1/718) /content/cars-moving.mp4: 384x640 2 cars, 1 truck, 437.2ms
video 1/1 (frame 2/718) /content/cars-moving.mp4: 384x640 2 cars, 1 truck, 186.3ms
video 1/1 (frame 3/718) /content/cars-moving.mp4: 384x640 1 bus, 2 trucks, 193.3ms
video 1/1 (frame 4/718) /content/cars-moving.mp4: 384x640 2 cars, 1 bus, 2 trucks, 178.0ms
video 1/1 (frame 5/718) /content/cars-moving.mp4: 384x640 1 person, 2 cars, 1 bus, 1 truck, 187.0ms
video 1/1 (frame

In [7]:
import cv2
from ultralytics import YOLO
from collections import defaultdict
import numpy as np

# Load the YOLOv8 model (pre-trained on COCO dataset)
model = YOLO("yolov8n.pt")  # "yolov8n" is the nano model; use "yolov8s.pt" or "yolov8m.pt" for better accuracy

# Path to your input video
video_path = "cars-moving.mp4"  # Update with your video path (e.g., "/content/drive/MyDrive/videos/input_video.mp4")
cap = cv2.VideoCapture(video_path)

# Get video properties
width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
fps = int(cap.get(cv2.CAP_PROP_FPS))

# Define output video writer
output_path = "output_video.mp4"
out = cv2.VideoWriter(output_path, cv2.VideoWriter_fourcc(*"mp4v"), fps, (width, height))

# Dictionary to store unique object counts by class
object_counts = defaultdict(int)
track_history = defaultdict(list)  # Track IDs to ensure unique counting

while cap.isOpened():
    ret, frame = cap.read()
    if not ret:
        break

    # Perform object detection and tracking
    results = model.track(frame, persist=True)  # Enable tracking with persist=True

    # Process detections
    for result in results:
        boxes = result.boxes.xyxy.cpu().numpy()  # Bounding boxes
        scores = result.boxes.conf.cpu().numpy()  # Confidence scores
        class_ids = result.boxes.cls.cpu().numpy().astype(int)  # Class IDs
        track_ids = result.boxes.id.cpu().numpy() if result.boxes.id is not None else None  # Track IDs

        if track_ids is not None:
            for box, score, class_id, track_id in zip(boxes, scores, class_ids, track_ids):
                if score > 0.5:  # Confidence threshold
                    class_name = model.names[class_id]
                    # Count unique objects based on track ID
                    if track_id not in track_history[class_name]:
                        object_counts[class_name] += 1
                        track_history[class_name].append(track_id)

                    # Draw bounding box and label
                    x1, y1, x2, y2 = map(int, box)
                    cv2.rectangle(frame, (x1, y1), (x2, y2), (0, 255, 0), 2)
                    label = f"{class_name} ID:{track_id} {score:.2f}"
                    cv2.putText(frame, label, (x1, y1 - 10), cv2.FONT_HERSHEY_SIMPLEX, 0.5, (0, 255, 0), 2)

    # Write the annotated frame to the output video
    out.write(frame)

# Release resources
cap.release()
out.release()

# Print object counts
print("Object Counts:")
for class_name, count in object_counts.items():
    print(f"{class_name}: {count}")

# Download the output video (optional)
from google.colab import files
files.download(output_path)

requirements: Ultralytics requirement ['lap>=0.5.12'] not found, attempting AutoUpdate...
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.7/1.7 MB 24.1 MB/s eta 0:00:00

requirements: AutoUpdate success ✅ 4.5s, installed 1 package: ['lap>=0.5.12']
WARNING ⚠️ requirements: Restart runtime or rerun command for updates to take effect


0: 384x640 2 cars, 1 truck, 163.1ms
Speed: 5.1ms preprocess, 163.1ms inference, 1.7ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 2 cars, 1 truck, 151.8ms
Speed: 2.9ms preprocess, 151.8ms inference, 1.6ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 1 car, 1 bus, 1 truck, 140.3ms
Speed: 3.0ms preprocess, 140.3ms inference, 1.3ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 1 car, 1 bus, 1 truck, 134.3ms
Speed: 3.3ms preprocess, 134.3ms inference, 1.4ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 2 cars, 1 truck, 134.6ms
Speed: 3.2ms preprocess, 134.6ms inference, 1.4ms postprocess per image at shap

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>